## EDA 3. 


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

In [2]:
# Load data
df = pd.read_csv('../data/processed/cleaned_data.csv')
alt = pd.read_csv('../data/references/alternative_service_free_youtube.csv')

In [3]:
df.shape

(5534, 21)

In [4]:
print(df['Churn'].value_counts())

Churn
No     4063
Yes    1471
Name: count, dtype: int64


### 3.1. Data Preparation

In [10]:

# Consistenct processing from EDA 01 and EDA 02
df['calculated_tenure'] = (df['TotalCharges'] / df['MonthlyCharges']).round().fillna(0).astype(int)

# Exclude July 2025 outlier (Validated EDA 01)
baseline = df[df['calculated_tenure'] !=7]

# Tenure segments (consistent with EDA 01)
def categorize_tenure(t):
    if t >= 5: return 'Pre-Inflection (5mo+)'   # Before Sep 2025
    elif t >= 3: return 'Inflection (3-5mo)'    # Sep-Oct 2025 - Policy shock period
    else: return 'Fresh (<3mo)'                 # After Nov 2025

df['Segment'] = df['calculated_tenure'].apply(categorize_tenure)

#Price Tier
df['PriceTier'] = pd.cut(df['MonthlyCharges'], bins=[0, 35, 75, 120], labels=['Low', 'Medium', 'High'])

# Family vs Individual (consistent with EDA 02)
df['is_family'] = (df['Partner'] == 'Yes' )| (df['Dependents'] == 'Yes')
df['segment_type'] = df['is_family'].map({True: 'Family', False: 'Individual'})

print(f"Segment distribution:\n{df['Segment'].value_counts()}")
print(f"\nFamily vs Individual:\n{df['segment_type'].value_counts()}")

Segment distribution:
Segment
Pre-Inflection (5mo+)    4598
Fresh (<3mo)              639
Inflection (3-5mo)        297
Name: count, dtype: int64

Family vs Individual:
segment_type
Family        2964
Individual    2570
Name: count, dtype: int64
